# 实验4.3 网络裁剪（Pruning）实验教程

> **运行环境**：cann_9.0.0-py3.11-A2-arm-20260715 | ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB
> **适用对象**：深度学习初学者
> **学习方式**：从上到下逐格运行，边学边练

---

## 🎯 学习目标

通过本 Notebook 你将学会：

1. 理解 **网络裁剪（Pruning）** 的基本原理与作用
2. 使用 PyTorch 内置 `torch.nn.utils.prune` 对 CNN 进行 **L1 非结构化裁剪**
3. 度量裁剪前后的 **稀疏率、精度、推理速度、模型大小**
4. 对裁剪后模型进行 **微调（Fine-tune）** 恢复精度
5. 结合 **量化感知训练（QAT）** 进一步压缩模型到 INT8
6. 在昇腾 910 NPU 上运行训练，并理解 NPU 与 CPU 在量化阶段的分工

## 📖 实验流程总览

```
定义模型 → FP32训练 → 保存原始模型 → L1裁剪 → 稀疏率检查
    → 裁剪效果评估 → 微调恢复精度 → QAT量化训练 → INT8转换 → 最终对比
```

> 💡 **提示**：请按顺序逐个运行代码格（Cell），观察每一步的输出，配合 Markdown 说明理解每一步在做什么。

## 一、什么是网络裁剪（Pruning）？

### 1.1 为什么需要裁剪？

深度神经网络通常存在大量 **冗余参数**：

- 很多权重值很小（接近 0），对输出贡献微弱
- 删除这些权重后，模型精度下降有限，但模型更小、更快

裁剪是模型压缩三大手段之一（裁剪 / 量化 / 知识蒸馏），目标是：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">目标</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">减小体积</td>
<td style="text-align: left;">降低存储与内存占用</td>
</tr>
<tr>
<td style="text-align: left;">加速推理</td>
<td style="text-align: left;">减少乘加运算次数</td>
</tr>
<tr>
<td style="text-align: left;">降低功耗</td>
<td style="text-align: left;">适合端侧 / 边缘部署</td>
</tr>
</table>

**裁剪目标详解**：
- **减小体积**：裁剪后零值权重如果用稀疏格式存储（如 CSR/CSC），可以跳过零值不存储，从而减小模型文件体积。但默认的稠密存储格式中零值仍占空间，体积不变。
- **加速推理**：如果硬件支持稀疏计算（如昇腾 NPU 的稀疏加速单元），可以跳过零值的乘加运算，真正减少计算量。标准硬件做稠密矩阵乘法时仍会计算 0×x=0，不会加速。
- **降低功耗**：减少计算量和数据搬运量直接降低能耗，对电池供电的端侧设备（如手机、IoT）尤为重要。

### 1.2 裁剪的分类

```
裁剪
├── 非结构化裁剪 (Unstructured)  ← 本实验使用
│   └── 将不重要的权重置零（稀疏矩阵）
└── 结构化裁剪 (Structured)
    └── 整个通道 / 滤波器删除（硬件友好）
```

### 1.3 L1 非结构化裁剪原理

**核心思想**：权重绝对值越小，对输出贡献越小，越可以裁掉。

**步骤**：

1. 对某层权重按绝对值排序
2. 将最小的 `amount` 比例的权重置为 0
3. 其余权重保持不变

例如 `amount=0.3` 表示把 30% 最接近 0 的权重清零。

> PyTorch 中对应 API：`torch.nn.utils.prune.l1_unstructured(module, name='weight', amount=0.3)`

## 二、环境准备

首先导入所需库，并检测运行设备。

- **昇腾 910 NPU**：通过 `torch_npu` 插件使用，训练阶段加速
- **CPU 回退**：量化转换阶段 PyTorch 仅支持 CPU，会自动切回 CPU

> ⚠️ 在 GitCode 昇腾 Notebook 中，`torch` 与 `torch_npu` 通常已预装。

In [ ]:
import torch
import torch.nn as nn
import torch.quantization
import torch.nn.utils.prune as prune
import os
import time
import copy

print('PyTorch 版本:', torch.__version__)

# ---- 设备检测：优先昇腾NPU，其次CUDA，最后CPU ----
try:
    import torch_npu  # 昇腾NPU插件
    device = torch.device('npu')
    print('✅ 检测到昇腾 NPU，训练将使用:', device)
except ImportError:
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print('✅ 检测到 CUDA GPU，训练将使用:', device)
    else:
        device = torch.device('cpu')
        print('ℹ️  未检测到NPU/GPU，训练将使用 CPU')

print('当前设备:', device)

## 三、定义模型

我们搭建一个用于 MNIST 风格手写数字识别的 **小型 CNN**：

```
输入(1×28×28)
  → QuantStub        # 量化入口标记
  → Conv2d(1→32, 3×3) → ReLU
  → Conv2d(32→64, 3×3) → ReLU
  → Flatten → Linear(64*28*28 → 10)
  → DeQuantStub       # 量化出口标记
输出(10类)
```

> `QuantStub` / `DeQuantStub` 是 PyTorch 量化框架要求的占位符，
> 在 FP32 训练时它们不做任何事；进入 QAT 阶段后会被替换为 FakeQuant 节点。

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # 量化占位符
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()

        # 卷积层
        self.conv1 = nn.Conv2d(1, 32, 3, 1, 1)   # 1→32通道
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)  # 32→64通道
        self.relu2 = nn.ReLU()

        # 全连接层
        self.fc = nn.Linear(64 * 28 * 28, 10)

    def forward(self, x):
        x = self.quant(x)
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)
        x = self.dequant(x)
        return x

print('✅ SimpleCNN 类定义完成')

In [ ]:
# 创建模型并迁移到设备
model = SimpleCNN().to(device)

print('===== 原始模型结构 =====')
print(model)

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
print(f'\n总参数量: {total_params:,}')

## 四、FP32 普通训练

先用随机数据做几轮 **FP32（32位浮点）训练**，让模型获得初始权重。

> 本实验重点演示裁剪流程，故用随机数据代替真实 MNIST 数据集。
> 实际项目中应替换为 `torchvision.datasets.MNIST` 的 DataLoader。

**代码说明**：
- 使用 SGD 优化器（学习率 0.01）和交叉熵损失函数。
- 共训练 5 个 epoch，每个 epoch 生成一个随机 batch（4 张 1×28×28 图，标签 0~9）。
- 前向→计算损失→清零梯度→反向→更新权重，标准训练流程。

**预期结果**：
- 打印 5 个 epoch 的 loss 值，由于使用随机数据，loss 不会有意义下降趋势，约在 2.0~2.5 波动（10 类的交叉熵随机基线为 ln(10)≈2.30）。

> **为什么 loss 约 2.3？** 交叉熵损失在 10 类均匀分布时的理论值为 ln(10)≈2.3026。随机标签下模型无法学到有用规律，loss 会在该值附近波动。


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

model.train()
print('===== 开始 FP32 训练 =====')

for epoch in range(5):
    # 随机模拟一个batch: 4张1×28×28灰度图, 标签0~9
    input_tensor = torch.randn(4, 1, 28, 28).to(device)
    target = torch.randint(0, 10, (4,)).to(device)

    output = model(input_tensor)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'epoch: {epoch}  loss: {loss.item():.4f}')

## 五、保存裁剪前模型（用于后续对比）

用 `copy.deepcopy` 保存一份裁剪前的模型副本，
后面会和裁剪后、微调后、量化后的模型做 **精度/速度/大小** 对比。

In [ ]:
model_before_prune = copy.deepcopy(model)
print('✅ 已保存裁剪前模型副本 model_before_prune')

## 六、模型裁剪（L1 非结构化裁剪）

### 6.1 裁剪函数说明

`apply_pruning` 会对模型中所有 `Conv2d` 和 `Linear` 层执行：

1. `prune.l1_unstructured(module, name='weight', amount=0.3)`
   - 按 L1 范数（绝对值）排序，把最小的 30% 权重置零
2. `prune.remove(module, 'weight')`
   - 把裁剪'永久化'：将掩码乘进权重，移除 hook，使裁剪后的零值固定

### 6.2 amount 参数

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">amount</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;">0.3</td>
<td style="text-align: left;">裁掉 30% 最小权重（本实验）</td>
</tr>
<tr>
<td style="text-align: left;">0.5</td>
<td style="text-align: left;">裁掉 50%（更激进，精度下降更多）</td>
</tr>
<tr>
<td style="text-align: left;">0.1</td>
<td style="text-align: left;">裁掉 10%（保守，精度影响小）</td>
</tr>
</table>

**amount 参数详解**：
- `amount` 表示要裁剪的权重比例，取值范围 [0, 1]。例如 `amount=0.3` 表示将每层权重按绝对值排序后，把最小的 30% 置零。
- 裁剪率越高，模型越稀疏，但精度下降风险也越大。一般从 0.1~0.3 开始尝试，配合微调恢复精度。
- 裁剪率与精度的关系不是线性的：少量裁剪（<30%）精度影响很小，但超过某个阈值后精度会急剧下降（因为开始裁掉重要的权重）。


In [ ]:
def apply_pruning(model, amount=0.3):
    """对模型中所有Conv2d和Linear层做L1非结构化裁剪"""
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            print(f'  裁剪层: {name}  (amount={amount})')
            prune.l1_unstructured(module, name='weight', amount=amount)
            # 使裁剪永久生效（把mask写回weight）
            prune.remove(module, 'weight')

print('===== 开始裁剪 =====')
apply_pruning(model, amount=0.3)
print('✅ 裁剪完成')

## 七、检查稀疏率

**稀疏率（Sparsity）** = 零值权重数 / 总权重数 × 100%

裁剪 30% 后，稀疏率应接近 30%，说明裁剪生效。

**预期结果**：
- `稀疏率 Sparsity: 约 30.00%`
- 例如 `零值 158000 / 总计 526000`（具体数字取决于模型参数量）

> **为什么稀疏率接近但不完全等于 30%？** `l1_unstructured` 对每层独立裁剪 30%，由于各层权重的绝对值分布不同，统计全模型稀疏率时可能略有偏差。此外，`prune.remove` 将掩码永久化后，零值固定在权重张量中，统计结果精确反映裁剪比例。


In [ ]:
def check_sparsity(model):
    """统计模型中Conv2d和Linear层的权重稀疏率"""
    total = 0
    zero = 0
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            w = m.weight.data
            total += w.numel()
            zero += (w == 0).sum().item()
    sparsity = 100.0 * zero / total
    print(f'稀疏率 Sparsity: {sparsity:.2f}%  (零值 {zero} / 总计 {total})')
    return sparsity

print('===== 裁剪后稀疏率 =====')
sparsity = check_sparsity(model)

## 七(补)、权重可视化——让裁剪"看得见"

光看数字不够直观，我们画出裁剪前后 conv1 权重分布直方图，
以及零值位置热力图，确认裁剪确实把小权重清零了。

In [ ]:
import matplotlib
matplotlib.use('Agg')  # notebook中可改为inline
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# ---- Top row: weight distribution histograms ----
w_before = model_before_prune.conv1.weight.data.cpu().numpy().flatten()
w_after  = model.conv1.weight.data.cpu().numpy().flatten()

axes[0,0].hist(w_before, bins=80, color='steelblue', alpha=0.8)
axes[0,0].set_title('Before Pruning: conv1 Weight Distribution')
axes[0,0].set_xlabel('Weight Value'); axes[0,0].set_ylabel('Count')

axes[0,1].hist(w_after, bins=80, color='coral', alpha=0.8)
axes[0,1].set_title('After Pruning: conv1 Weight Distribution (spike at 0)')
axes[0,1].set_xlabel('Weight Value'); axes[0,1].set_ylabel('Count')

# ---- Bottom row: zero-location heatmap (white=zero, black=nonzero) ----
mask_before = (model_before_prune.conv1.weight.data.cpu().abs() > 1e-8).numpy()
mask_after  = (model.conv1.weight.data.cpu().abs() > 1e-8).numpy()

# Show the 2D slice of the 0th filter
axes[1,0].imshow(mask_before[0,0], cmap='gray')
axes[1,0].set_title('Before Pruning: conv1[0,0] Nonzero Locations (all black = all kept)')

axes[1,1].imshow(mask_after[0,0], cmap='gray')
axes[1,1].set_title('After Pruning: conv1[0,0] Nonzero Locations (white dots = pruned to zero)')

plt.tight_layout()
plt.show()

print(f'conv1 weights: nonzero before pruning = {mask_before.sum()}, nonzero after pruning = {mask_after.sum()}')

## 八、裁剪效果评估

定义三个评估工具函数，对比裁剪前后：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">函数</th>
<th style="text-align: left;">作用</th>
</tr>
<tr>
<td style="text-align: left;"><code>evaluate(model, test_data)</code></td>
<td style="text-align: left;">在 <strong>固定</strong> 测试集上计算准确率</td>
</tr>
<tr>
<td style="text-align: left;"><code>measure_time(model)</code></td>
<td style="text-align: left;">测 200 次推理总耗时</td>
</tr>
<tr>
<td style="text-align: left;"><code>get_model_size(model)</code></td>
<td style="text-align: left;">获取模型权重文件大小(KB)</td>
</tr>
</table>

**评估函数说明**：
- `evaluate`：在固定的 100 个测试样本上计算准确率。使用固定测试集是为了保证裁剪前后的对比是公平的——如果每次评估用不同随机数据，精度差异可能来自数据而非裁剪。
- `measure_time`：先做 10 次预热推理（消除首次推理的编译开销），然后计时 200 次推理。NPU 上需要调用 `synchronize()` 确保异步计算完成。
- `get_model_size`：将模型 `state_dict` 保存到临时文件并读取文件大小，反映模型的存储开销。

> ⚠️ **关键改进**：评估时使用 **同一批固定随机样本** (`test_data`)，
> 保证裁剪前后的对比是公平的——否则两次评估用不同数据，结果无法对比。

### 关于模型大小和推理时间"没变化"的说明

你可能发现裁剪后 **模型大小和推理时间几乎不变**，这是正常的！

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">现象</th>
<th style="text-align: left;">原因</th>
</tr>
<tr>
<td style="text-align: left;">模型大小不变</td>
<td style="text-align: left;"><strong>非结构化裁剪</strong>只是把权重值改为 0.0，零值仍以 FP32 存储，文件体积不变。要真正减小体积需用稀疏存储格式或结构化裁剪</td>
</tr>
<tr>
<td style="text-align: left;">推理时间不变</td>
<td style="text-align: left;">标准硬件做稠密矩阵乘法，仍会计算所有 0×x，不会跳过零值。需要专用稀疏计算内核才能加速</td>
</tr>
<tr>
<td style="text-align: left;">精度有变化</td>
<td style="text-align: left;">这才是非结构化裁剪最直接的影响——少量权重被清零，输出略有改变</td>
</tr>
</table>

**"没变化"现象的深层原因**：
- **模型大小不变**：非结构化裁剪只是把权重值改为 0.0，但 0.0 仍以 FP32 格式（4 字节）存储在 `state_dict` 中。要真正减小体积，需要使用稀疏存储格式（如 PyTorch 的 `to_sparse()`）或进行结构化裁剪（删除整个通道）。
- **推理时间不变**：标准硬件（CPU/NPU）的矩阵乘法是稠密的，即遍历所有元素计算 `C[i,j] += A[i,k] * B[k,j]`，即使 `A[i,k]=0` 也会执行乘法和加法。只有专用稀疏计算引擎才能跳过零值。
- **精度有变化**：被裁掉的权重虽然绝对值小，但对输出仍有微小贡献。清零后输出会略有改变，可能导致部分样本的预测类别翻转，从而影响准确率。

> 💡 非结构化裁剪的价值在于：配合 **量化** 后，稀疏的 INT8 权重可被专用硬件/引擎（如昇腾 ACL）跳过零值计算，从而真正加速。

In [ ]:
# ---- 生成固定的测试数据集，保证裁剪前后用同一批样本对比 ----
torch.manual_seed(42)  # 固定随机种子，保证可复现
test_data = [(torch.randn(1, 1, 28, 28), torch.randint(0, 10, (1,)))
             for _ in range(100)]
print(f'已生成 {len(test_data)} 个固定测试样本 (seed=42)')


def evaluate(model, test_data, device=None):
    """在固定测试集上评估准确率，保证公平对比"""
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    correct = 0
    total = len(test_data)
    with torch.no_grad():
        for x, y in test_data:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
    return correct / total


def measure_time(model, device=None):
    """测量200次推理总耗时(秒)"""
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    x = torch.randn(1, 1, 28, 28).to(device)
    # 预热
    with torch.no_grad():
        for _ in range(10):
            model(x)
    if device.type == 'npu':
        torch.npu.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(200):
            model(x)
    if device.type == 'npu':
        torch.npu.synchronize()
    end = time.time()
    return end - start


def get_model_size(model):
    """返回模型state_dict文件大小(KB)"""
    torch.save(model.state_dict(), 'temp.p')
    size = os.path.getsize('temp.p') / 1024
    os.remove('temp.p')
    return size

print('评估函数定义完成')

In [ ]:
print('===== 裁剪效果评估 (使用固定测试集) =====')

# 用同一批test_data评估两个模型，保证公平
acc_before = evaluate(model_before_prune, test_data, device)
acc_after_prune = evaluate(model, test_data, device)

time_before = measure_time(model_before_prune, device)
time_after_prune = measure_time(model, device)

size_before = get_model_size(model_before_prune)
size_after = get_model_size(model)

print(f'精度:     裁剪前={acc_before:.2f}  裁剪后={acc_after_prune:.2f}  (变化={acc_after_prune-acc_before:+.2f})')
print(f'推理时间: 裁剪前={time_before:.4f}s  裁剪后={time_after_prune:.4f}s')
print(f'模型大小: 裁剪前={size_before:.2f}KB  裁剪后={size_after:.2f}KB  (非结构化裁剪体积不变是正常的)')
print(f'稀疏率:   {sparsity:.2f}%')

# 逐层对比裁剪前后输出差异，证明裁剪确实改变了模型行为
sample_x = test_data[0][0].to(device)
with torch.no_grad():
    out_before = model_before_prune(sample_x)
    out_after  = model(sample_x)
diff = (out_before - out_after).abs().mean().item()
print(f'\n同一样本输出差异(裁剪前-裁剪后): {diff:.6f}  (>0说明裁剪改变了模型输出)')

## 九、裁剪后微调（Fine-tune）

裁剪会损失一部分精度，通过 **少量轮次微调** 让剩余权重重新适应任务，恢复精度。

> 微调时被裁剪为零的权重 **仍可更新**（因为 `prune.remove` 已移除掩码约束）。
> 如需保持稀疏结构，可使用 `prune` 的永久掩码模式（本实验简化处理）。

In [ ]:
print('===== 裁剪后微调 =====')
model.train()

for epoch in range(3):
    input_tensor = torch.randn(4, 1, 28, 28).to(device)
    target = torch.randint(0, 10, (4,)).to(device)

    output = model(input_tensor)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'finetune epoch: {epoch}  loss: {loss.item():.4f}')

acc_after_finetune = evaluate(model, test_data, device)
print(f'\n微调后精度: {acc_after_finetune:.2f}  (裁剪后={acc_after_prune:.2f})')

## 十、量化感知训练（QAT）

### 10.1 为什么还要量化？

裁剪让权重变稀疏（多出很多零），但权重仍以 **FP32** 存储。
量化把权重从 FP32 压缩到 **INT8**，体积再降约 4 倍。

```
裁剪(稀疏30%) + 量化(INT8) = 双重压缩
```

### 10.2 QAT vs PTQ

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方式</th>
<th style="text-align: left;">全称</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;">QAT</td>
<td style="text-align: left;">Quantization-Aware Training</td>
<td style="text-align: left;">训练时插入FakeQuant，精度更高（本实验）</td>
</tr>
<tr>
<td style="text-align: left;">PTQ</td>
<td style="text-align: left;">Post-Training Quantization</td>
<td style="text-align: left;">训练后直接量化，简单但精度略低</td>
</tr>
</table>

**QAT vs PTQ 详解**：
- **QAT（量化感知训练）**：在训练过程中插入 FakeQuant 节点，前向传播时模拟 INT8 量化/反量化的舍入误差，但反向传播仍用浮点计算。模型在微调中逐渐适应量化噪声，学出对量化更鲁棒的权重。最终转换后的 INT8 模型精度通常显著优于 PTQ。
- **PTQ（训练后量化）**：直接将已训练好的 FP32 模型转换为 INT8，不需要额外训练。实现简单快速，但模型从未"见过"量化噪声，精度损失不可控。
- **本实验选择 QAT** 的原因：裁剪后模型已经过微调，再用 QAT 可以让模型同时适应裁剪和量化两种压缩，精度损失最小。

### 10.3 ⚠️ 昇腾平台注意事项

PyTorch 官方量化（`prepare_qat` / `convert`）仅支持 **CPU 后端**。
因此本阶段需要把模型迁回 CPU 进行量化配置与转换。

In [ ]:
# ---- 量化需在CPU上进行 ----
model = model.cpu()
device_cpu = torch.device('cpu')
print('模型已迁回 CPU 以进行量化')

# 选择量化后端引擎
engine = torch.backends.quantized.supported_engines[0]
torch.backends.quantized.engine = engine
print(f'量化后端引擎: {engine}')

# 设置QAT量化配置
model.qconfig = torch.quantization.get_default_qat_qconfig(engine)
print(f'QAT qconfig: {model.qconfig}')

In [ ]:
# 插入FakeQuant节点
model.train()
torch.quantization.prepare_qat(model, inplace=True)

print('===== 插入FakeQuant后的模型 =====')
print(model)

In [ ]:
print('===== QAT 训练 =====')
model.train()

for epoch in range(3):
    input_tensor = torch.randn(4, 1, 28, 28)
    target = torch.randint(0, 10, (4,))

    output = model(input_tensor)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'QAT epoch: {epoch}  loss: {loss.item():.4f}')

## 十一、转换为 INT8 量化模型

`torch.quantization.convert` 把训练时的 FakeQuant 节点替换为真正的 **量化算子**，
权重由 FP32 变为 INT8（`torch.qint8` / `torch.quint8`）。

In [ ]:
model.eval()
torch.quantization.convert(model, inplace=True)

print('===== 量化后模型 =====')
print(model)

## 十二、INT8 模型推理测试

用一张随机图片测试量化后模型能否正常输出 10 类 logits。

In [ ]:
input_tensor = torch.randn(1, 1, 28, 28)

with torch.no_grad():
    output = model(input_tensor)

print('量化后输出:')
print(output)
print(f'\n预测类别: {output.argmax(dim=1).item()}')

## 十三、模型大小与权重类型

观察量化后模型体积是否减小、权重 dtype 是否变为 INT8。

In [ ]:
def print_model_size(model):
    torch.save(model.state_dict(), 'temp.p')
    size = os.path.getsize('temp.p') / 1024
    print(f'模型大小: {size:.2f} KB')
    os.remove('temp.p')
    return size

print('===== 模型大小 =====')
size_int8 = print_model_size(model)

print('\n===== 权重类型 =====')
print('conv1 权重 dtype:', model.conv1.weight().dtype)

## 十四、最终效果对比总结

把 **裁剪前 → 裁剪后 → 微调后 → 量化后(INT8)** 四个阶段的
精度、推理时间、模型大小汇总对比，直观感受裁剪+量化的压缩效果。

**预期结果说明**：
- **精度对比**：裁剪前≈裁剪后（随机数据下精度约 0.05~0.15，10 类随机猜测）；微调后可能略有变化；量化后(INT8)精度与微调后接近。
- **推理时间对比**：裁剪前≈裁剪后（非结构化裁剪不加速）；量化后(INT8)可能略快（INT8 运算在 CPU 上可能更快，但差异不大）。
- **模型大小对比**：裁剪前≈裁剪后（零值仍以 FP32 存储）；量化后(INT8)约为裁剪前的 1/4（权重从 4 字节降为 1 字节）。
- **稀疏率**：约 30%（裁剪比例）。

> **核心结论**：非结构化裁剪产生稀疏权重但体积不变；量化(INT8)才真正减小体积。两者结合（稀疏+低比特）才能在专用硬件上同时获得稀疏加速和低比特加速。


In [ ]:
# 量化后模型在CPU上，test_data也用于CPU评估
acc_int8 = evaluate(model, test_data, device_cpu)
time_int8 = measure_time(model, device_cpu)

print('=' * 55)
print('                 最终对比总结')
print('=' * 55)

print('\n【精度对比】(同一固定测试集, 公平对比)')
print(f'  裁剪前(FP32):      {acc_before:.2f}')
print(f'  裁剪后:            {acc_after_prune:.2f}')
print(f'  微调后:            {acc_after_finetune:.2f}')
print(f'  量化后(INT8):      {acc_int8:.2f}')

print('\n【推理时间对比】(200次)')
print(f'  裁剪前:            {time_before:.4f}s')
print(f'  裁剪后:            {time_after_prune:.4f}s')
print(f'  量化后(INT8):      {time_int8:.4f}s  ← 量化后体积变小,速度可能更快')

print('\n【模型大小对比】')
print(f'  裁剪前(FP32):      {size_before:.2f} KB')
print(f'  裁剪后(FP32):      {size_after:.2f} KB  ← 非结构化裁剪体积不变')
print(f'  量化后(INT8):      {size_int8:.2f} KB  ← 量化后体积显著减小')

print('\n【稀疏率】')
print(f'  裁剪后稀疏率:      {sparsity:.2f}%')

print('=' * 55)
print('\n💡 结论: 非结构化裁剪→稀疏(体积不变) + 量化→INT8(体积变小)')
print('   二者结合才能同时获得稀疏性和小体积,便于昇腾等硬件加速部署。')

## 十五、学习总结与思考

### ✅ 本实验你完成了

1. **定义** 带量化占位符的 SimpleCNN
2. **FP32 训练** 获得初始权重
3. **L1 非结构化裁剪** 30% 权重 → 验证稀疏率
4. **评估** 裁剪前后精度 / 速度 / 体积
5. **微调** 恢复裁剪损失的精度
6. **QAT 量化感知训练** → **INT8 转换**
7. **全流程对比** 裁剪前 vs 裁剪后 vs 微调后 vs 量化后

### 🤔 课后思考

1. 把 `amount` 改成 `0.5` 或 `0.7`，观察精度下降幅度——**裁剪率与精度的权衡**
2. 把 `l1_unstructured` 换成 `ln_structured`（结构化裁剪），对比模型大小是否变化？
3. 为什么非结构化裁剪后模型大小几乎不变？（提示：零值仍以FP32占存储，需稀疏格式）
4. 为什么非结构化裁剪后推理时间几乎不变？（提示：稠密矩阵乘法仍计算0×x）
5. 在昇腾 910 上，如何用 MindSpore 或 ACL 把量化模型真正部署加速？
6. 裁剪 + 量化 + 知识蒸馏三者如何组合使用？

### 🔑 核心结论

> - **非结构化裁剪**：产生稀疏权重，但模型体积和推理速度不变，需配合稀疏计算引擎
> - **量化(INT8)**：真正减小模型体积(~4x)，是本实验中体积下降的来源
> - **裁剪+量化组合**：既稀疏又低比特，昇腾等NPU硬件可进一步加速

### 📚 延伸阅读

- PyTorch Pruning 教程：https://pytorch.org/tutorials/intermediate/pruning_tutorial.html
- 论文：*Learning both Weights and Connections for Efficient Neural Networks* (Han et al., 2015)
- 昇腾开发文档：https://www.hiascend.com/document

---

> 🎉 恭喜完成网络裁剪实验！请尝试修改参数重新运行，加深理解。

---

## 课后练习

请根据本实验内容完成以下题目进行自测。


**第1题**（单选题）网络裁剪的主要目标是？

- A. 增加模型精度
- B. 减小体积、加速推理、降低功耗
- C. 增加参数量
- D. 提高训练速度


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）L1 非结构化裁剪的核心思想是？

- A. 删除整个通道
- B. 将绝对值最小的权重置零
- C. 降低数据精度
- D. 增加网络层数


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）amount=0.3 表示什么？

- A. 裁掉 70% 权重
- B. 裁掉 30% 最小权重
- C. 保留 30% 权重
- D. 裁剪 3 层


In [ ]:
q3 = ''  # 填入你的选项，如 'A'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）非结构化裁剪后模型大小为什么几乎不变？

- A. 裁剪没有效果
- B. 零值仍以 FP32 存储，需稀疏格式
- C. 裁剪率太低
- D. 模型太小


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）非结构化裁剪后推理时间为什么几乎不变？

- A. 裁剪无效
- B. 稠密矩阵乘法仍计算 0×x
- C. NPU 太快
- D. 数据量太小


In [ ]:
q5 = ''  # 填入你的选项，如 'B'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）裁剪后微调的作用是？

- A. 进一步减小模型
- B. 恢复裁剪损失的精度
- C. 改变模型结构
- D. 加速推理


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）PyTorch 中 L1 非结构化裁剪的 API 是？

- A. torch.prune_l1()
- B. torch.nn.utils.prune.l1_unstructured
- C. torch.l1_prune()
- D. prune.l1()


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）非结构化裁剪 vs 结构化裁剪，哪个对硬件更友好？

- A. 非结构化裁剪
- B. 结构化裁剪（整个通道/滤波器删除）
- C. 都一样
- D. 都不友好


In [ ]:
q8 = ''  # 填入你的选项，如 'A'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）裁剪 + 量化的组合效果是？

- A. 只减小体积
- B. 既稀疏又低比特，硬件可进一步加速
- C. 只加速推理
- D. 没有额外效果


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）稀疏率的定义是？

- A. 非零权重数 / 总权重数
- B. 零值权重数 / 总权重数 × 100%
- C. 总权重数 / 零值数
- D. 裁剪的层数


In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_05 import grade
grade(globals())

## 参考资料

- [PyTorch Pruning 教程](https://pytorch.org/tutorials/intermediate/pruning_tutorial.html)
- [昇腾社区文档](https://hiascend.com/document)
